# HadithMisinfoBench — audit & re-analysis notebook
Runs on Colab (GPU recommended) or locally. It reads the data and the cached Qwen translations straight from the public repo, rebuilds every retrieval system, and produces the numbers the paper currently lacks.

**What it does**
1. **Data audit** — reproduces the gold-provenance finding (gold = BM25 top-5 of the Arabic matn), English-text coverage, duplicates, and gold quality.
2. **Rebuilds all 11 systems** (English/Bangla BM25, Qwen→BM25, 4 dense models, 4 RRF fusions) **plus 4 Bangla-only RRF systems S12–S15** (Qwen translation of the Bangla claim + dense; no English query) and **saves per-claim top-10 rankings** (the repo never stored them).
3. **Reproduction check** against `results/tables/retrieval_summary_final.csv`.
4. **Bootstrap, claim-level and Matn-clustered**, for every Table IV comparison plus the RRF comparisons the paper lacks CIs for.
5. **Sensitivity subsets**: unique Matns only, reliable-gold only (containment ≥ 0.9), claims whose gold has English text (and those whose gold has none), and a twin-aware gold variant.
6. **RRF depth ablation** (paper fuses top-10 lists; here also 20/50/100), plus a Bangla-only RRF (Qwen translation + dense) that needs no English query.
7. **Failure diagnostics** for the MiniLM section: modal top-1 share per system on own and *shared* failures, hubness skew, and an EN–BN embedding-alignment check (tests "Bangla unsupported" vs "hubness").
8. **Figure data export (Fig. 4 and Fig. 5)** — saves the underlying CSV tables for plotting externally (no figures rendered), then a zip of everything.

**Runtime (T4):** about 30–50 min the first time (document embeddings for 4 models are cached to disk). Set `RUN_DENSE = False` for a CPU-only run of everything except the dense/RRF/diagnostic parts (about 1 minute).

**Deliberate differences from the paper's notebook:** BM25 ties are broken by corpus order (stable sort) so results are deterministic. Direct Bangla BM25 therefore differs slightly from the paper's 0.3% (968 of 986 Bangla queries have no matching token, so its top-10 is pure tie order). Dense scores can differ in the 3rd decimal from fp16 / hardware. BM25 is a sparse-matrix implementation that is numerically identical to `rank_bm25.BM25Okapi` (checked in the next cells) but about 100× faster.

In [ ]:
%pip -q install sentence-transformers scipy pandas

In [ ]:
import os, re, json, math, gc, csv, hashlib, random, shutil, subprocess, unicodedata, collections, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
SEED = 42
N_BOOT = 2000
RUN_DENSE = True            # False = CPU-only (BM25 + translation only)
RRF_K = 60
TOP_K = 10                  # paper setting
DEEP_K = 100                # depth used for the RRF sensitivity run
CONTAINMENT_THRESHOLD = 0.9
REPO_URL = "https://github.com/Rahatut/hadith-misinfo-bench.git"
WORK = Path("hmb_reanalysis"); WORK.mkdir(exist_ok=True)
OUT = WORK / "outputs"; OUT.mkdir(exist_ok=True)
FIG = OUT / "figure_data"; FIG.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print("work dir:", WORK.resolve())

In [ ]:
# ---- 1. Get data (public repo). Set HMB_REPO to use a local checkout instead. ----
REPO = Path(os.environ.get("HMB_REPO", str(WORK / "repo")))
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
EV_PATH = REPO / "data/processed/evidence.jsonl"
BM_PATH = REPO / "data/processed/benchmark/benchmark_dataset_a.jsonl"
TR_PATH = REPO / "results/cache/bangla_to_english_qwen25_3b_seed42.json"
PAPER_TABLE = REPO / "results/tables/retrieval_summary_final.csv"
for p in (EV_PATH, BM_PATH, TR_PATH, PAPER_TABLE):
    assert p.exists(), f"missing {p}"
print("repo OK:", REPO)

In [ ]:
def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

evidence = read_jsonl(EV_PATH)
bench = read_jsonl(BM_PATH)
ev_ids = [e["evidence_id"] for e in evidence]
ev_idx = {i: j for j, i in enumerate(ev_ids)}
N_DOC = len(evidence)

def retrieval_text(e):
    """Arabic Matn + English translation (when present) -- the evidence representation used in the paper."""
    return "\n".join([e["arabic_matn"]] + ([e["english_text"]] if e.get("english_text") else []))

auth = [r for r in bench if r["label"] == "authentic" and r["gold_evidence_ids"]]
Q = len(auth)
claim_ids = [r["claim_id"] for r in auth]
q_en = [r["claims"]["en"] for r in auth]
q_bn = [r["claims"]["bn"] for r in auth]
q_ar = [r["canonical"]["ar"] for r in auth]

sha24 = lambda t: hashlib.sha256((t or "").encode("utf-8")).hexdigest()[:24]
translation_cache = json.loads(TR_PATH.read_text(encoding="utf-8"))
missing = [r["claim_id"] for r in auth if sha24(r["claims"]["bn"]) not in translation_cache]
assert not missing, f"{len(missing)} claims lack a cached Qwen translation"
q_qwen = [translation_cache[sha24(b)] for b in q_bn]

gold_idx = [np.array([ev_idx[g] for g in r["gold_evidence_ids"]]) for r in auth]
gold_mask = np.zeros((Q, N_DOC), bool)
for i, g in enumerate(gold_idx):
    gold_mask[i, g] = True

print(f"evidence records: {N_DOC:,} | benchmark records: {len(bench):,} | authentic with gold: {Q}")
if Q != 986:
    print("WARNING: expected 986 authentic claims with gold (paper); found", Q)

## 0. Utilities
* `FastBM25`: sparse BM25 numerically identical to `rank_bm25.BM25Okapi` (k1=1.5, b=0.75, ε=0.25), deterministic stable tie-breaking.
* `tok_exp`: the tokenizer used by the paper's experiment notebook (diacritics kept) — used for all 11 systems.
* `normalize_and_tokenize`: the repo's own normaliser (diacritics stripped, alef unified) — this is what built the gold labels, so it is used for the provenance audit.
* Metrics, claim-level and cluster bootstrap.

In [ ]:
import sys
from scipy import sparse
from scipy import stats as sps

sys.path.insert(0, str((REPO / "src").resolve()))
from hadith_misinfo.preprocessing.normalize import normalize_and_tokenize   # repo normaliser (built the gold)

def tok_exp(text):
    return re.findall(r"[\w\u0600-\u06FF\u0980-\u09FF]+", (text or "").lower())   # paper experiment tokenizer

class FastBM25:
    """Sparse BM25Okapi. Same idf/epsilon handling and query-token multiplicity as rank_bm25."""
    def __init__(self, docs_tokens, k1=1.5, b=0.75, epsilon=0.25):
        vocab, rows, cols, vals = {}, [], [], []
        dl = np.zeros(len(docs_tokens))
        for d, toks in enumerate(docs_tokens):
            dl[d] = len(toks)
            for t, n in collections.Counter(toks).items():
                rows.append(vocab.setdefault(t, len(vocab))); cols.append(d); vals.append(n)
        V, N = len(vocab), len(docs_tokens)
        tf = sparse.coo_matrix((vals, (rows, cols)), shape=(V, N)).tocsr()
        df = np.diff(tf.indptr).astype(float)
        idf = np.log(N - df + 0.5) - np.log(df + 0.5)
        avg_idf = idf.sum() / V
        idf[idf < 0] = epsilon * avg_idf
        avgdl = dl.sum() / N
        norm = k1 * (1 - b + b * dl / avgdl)
        c = tf.tocoo()
        data = idf[c.row] * c.data * (k1 + 1) / (c.data + norm[c.col])
        self.W = sparse.csr_matrix((data, (c.row, c.col)), shape=(V, N))
        self.vocab, self.N = vocab, N

    def scores(self, qtoks_list):
        rows, cols, vals = [], [], []
        for i, q in enumerate(qtoks_list):
            for t, n in collections.Counter(t for t in q if t in self.vocab).items():
                rows.append(i); cols.append(self.vocab[t]); vals.append(float(n))
        Qm = sparse.csr_matrix((vals, (rows, cols)), shape=(len(qtoks_list), len(self.vocab)))
        return (Qm @ self.W).toarray()

    def search(self, qtoks_list, k=100, chunk=200):
        ranks = []
        for s in range(0, len(qtoks_list), chunk):
            S = self.scores(qtoks_list[s:s + chunk])
            ranks.append(np.argsort(-S, axis=1, kind="stable")[:, :k].astype(np.int32))
        return np.vstack(ranks)

# ---- metrics (rankings are int arrays: queries x depth, columns = evidence index) ----
def hit_matrix(ranks, mask):
    return mask[np.arange(len(ranks))[:, None], ranks]

def metrics(ranks, mask, rows=None):
    if rows is not None:
        ranks, mask = ranks[rows], mask[rows]
    h = hit_matrix(ranks, mask)[:, :TOP_K]
    first = np.where(h.any(1), h.argmax(1) + 1, 0)
    return {"R@1": h[:, :1].any(1).mean(), "R@5": h[:, :5].any(1).mean(), "R@10": h.any(1).mean(),
            "MRR": np.where(first > 0, 1.0 / np.maximum(first, 1), 0.0).mean()}

def hit_vec(ranks, mask, k, rows=None):
    if rows is not None:
        ranks, mask = ranks[rows], mask[rows]
    return hit_matrix(ranks, mask)[:, :k].any(1).astype(float)

# ---- bootstrap: claim-level or cluster-level (resample whole clusters of claims) ----
def paired_diff_ci(a, b, clusters=None, n_boot=N_BOOT, seed=SEED):
    """Paired difference mean(b) - mean(a) with 95% percentile CI. a=zeros gives a CI for b's mean."""
    d = np.asarray(b, float) - np.asarray(a, float)
    rng = np.random.default_rng(seed)
    if clusters is None:
        idx = rng.integers(0, len(d), (n_boot, len(d)))
        boots = d[idx].mean(1)
    else:
        _, inv = np.unique(clusters, return_inverse=True)
        C = inv.max() + 1
        s = np.bincount(inv, weights=d, minlength=C)
        m = np.bincount(inv, minlength=C).astype(float)
        idx = rng.integers(0, C, (n_boot, C))
        boots = s[idx].sum(1) / m[idx].sum(1)
    lo, hi = np.quantile(boots, [0.025, 0.975])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return float(d.mean()), float(lo), float(hi), float(min(p, 1.0))

def show(df, **kw):
    try:
        display(df)            # Jupyter / Colab
    except NameError:
        print(df.to_string(index=False))

In [ ]:
# ---- equivalence test: FastBM25 vs the reference rank_bm25 on a few real queries ----
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "rank_bm25"], check=True)
    from rank_bm25 import BM25Okapi
corpus_tok = [tok_exp(retrieval_text(e)) for e in evidence]
fast = FastBM25(corpus_tok)
ref = BM25Okapi(corpus_tok)
probe = [tok_exp(q) for q in q_en[:10] + q_bn[:5]]
S_fast = fast.scores(probe)
max_diff = max(np.abs(S_fast[i] - ref.get_scores(q)).max() for i, q in enumerate(probe))
print("max |FastBM25 - rank_bm25| over 15 probe queries:", max_diff)
assert max_diff < 1e-6
del ref; gc.collect()

## 1. Data audit
Every number below is computed from the repo files; nothing is typed in by hand.

In [ ]:
# ---- 1a. Label counts vs. what the paper says ----
AUDIT = {}
n_auth_all = sum(r["label"] == "authentic" for r in bench)
n_fab_all = sum(r["label"] == "fabricated" for r in bench)
gold_len = collections.Counter(len(r["gold_evidence_ids"]) for r in bench if r["label"] == "authentic")
AUDIT.update(n_records=len(bench), n_authentic=n_auth_all, n_fabricated=n_fab_all,
             n_authentic_with_gold=Q, gold_ids_per_claim=dict(gold_len))
print(f"Paper: 2,000 records = 1,000 authentic + 1,000 fabricated; 986 authentic have gold, 14 excluded.")
print(f"File : {len(bench):,} records = {n_auth_all} authentic + {n_fab_all} fabricated; authentic with gold = {Q}")
print("Gold IDs per authentic claim:", dict(gold_len))

In [ ]:
# ---- 1b. Gold provenance: re-run the repo's own BM25 (repo normaliser, Arabic+English index) on the Arabic matn ----
t_norm = [normalize_and_tokenize(retrieval_text(e)) for e in evidence]
prov_bm25 = FastBM25(t_norm)
prov_ranks = prov_bm25.search([normalize_and_tokenize(a) for a in q_ar], k=5)
stored_ok = all(len(g) == 5 for g in gold_idx)
exact = np.mean([stored_ok and np.array_equal(prov_ranks[i], gold_idx[i]) for i in range(Q)])
as_set = np.mean([set(prov_ranks[i].tolist()) == set(gold_idx[i].tolist()) for i in range(Q)])
top1 = np.mean([prov_ranks[i][0] == gold_idx[i][0] for i in range(Q)])
AUDIT.update(gold_reproduced_exact_ordered=float(exact), gold_reproduced_as_set=float(as_set), gold_top1_reproduced=float(top1))
print(f"Stored gold == BM25 top-5 of the Arabic matn:  exact order {exact:.1%} | as a set {as_set:.1%} | top-1 {top1:.1%}")
print("=> gold labels are the output of a BM25 search over this same corpus, not an independent identifier match.")

# collection skew introduced by that procedure
coll = lambda i: ev_ids[i].split("_")[0]
corp_share = collections.Counter(coll(j) for j in range(N_DOC))
gold1_share = collections.Counter(coll(g[0]) for g in gold_idx)
print("\ncollection share  (corpus vs. top-1 gold):")
for c in sorted(corp_share):
    print(f"  {c:8s} corpus {corp_share[c]/N_DOC:6.1%} | top-1 gold {gold1_share[c]/Q:6.1%}")
AUDIT["gold_top1_collection_share"] = {c: gold1_share[c] / Q for c in gold1_share}

In [ ]:
# ---- 1c. English-text coverage (the evidence text is Arabic + English *when available*) ----
has_en = np.array([bool(e.get("english_text")) for e in evidence])
print(f"corpus records with English text: {has_en.mean():.1%}")
for c in sorted(corp_share):
    sel = np.array([coll(j) == c for j in range(N_DOC)])
    print(f"  {c:8s}: {has_en[sel].mean():6.1%} have English")
gold_has_en = np.array([[has_en[j] for j in g] for g in gold_idx])           # Q x 5
cl_any, cl_all, cl_top1 = gold_has_en.any(1), gold_has_en.all(1), gold_has_en[:, 0]
print(f"\nclaims whose top-1 gold has English: {cl_top1.mean():.1%} | any of 5 gold: {cl_any.mean():.1%} | all 5: {cl_all.mean():.1%}")
print("An English-text query can only match English text; compare these shares with English BM25's R@1/R@10 in section 3.")
AUDIT.update(corpus_english_share=float(has_en.mean()), claims_top1_gold_english=float(cl_top1.mean()),
             claims_any_gold_english=float(cl_any.mean()), claims_all_gold_english=float(cl_all.mean()))

In [ ]:
# ---- 1d. Duplicates ----
matn_key = [" ".join(normalize_and_tokenize(a)) for a in q_ar]
_, cluster_id = np.unique(matn_key, return_inverse=True)
csize = collections.Counter(cluster_id.tolist())
n_unique_matn = len(csize)
print(f"authentic claims: {Q} | unique source Matns: {n_unique_matn} | claims that repeat an earlier Matn: {Q - n_unique_matn}")
print("cluster sizes:", dict(collections.Counter(csize.values())))
dup_bn = Q - len(set(q_bn)); dup_en = Q - len(set(q_en))
print(f"duplicate Bangla claim texts: {dup_bn} | duplicate English claim texts: {dup_en}")

ar_key = [" ".join(normalize_and_tokenize(e["arabic_matn"])) for e in evidence]
groups = collections.defaultdict(list)
for j, k in enumerate(ar_key):
    groups[k].append(j)
n_dup_ev = N_DOC - len(groups)
print(f"evidence records whose normalised Arabic text duplicates another record: {n_dup_ev} (unique texts: {len(groups):,})")
twin_exp = [np.unique(np.concatenate([groups[ar_key[j]] for j in g])) for g in gold_idx]
n_with_twin = sum(len(t) > len(g) for t, g in zip(twin_exp, gold_idx))
print(f"claims whose gold set has an identical-text twin outside it: {n_with_twin} ({n_with_twin/Q:.1%})")
twin_mask = np.zeros_like(gold_mask)
for i, t in enumerate(twin_exp):
    twin_mask[i, t] = True
AUDIT.update(n_unique_matns=n_unique_matn, n_repeat_matn_claims=Q - n_unique_matn, dup_bn_claims=dup_bn, dup_en_claims=dup_en,
             evidence_duplicate_texts=n_dup_ev, claims_with_gold_twin=int(n_with_twin))

In [ ]:
# ---- 1e. Gold quality: how much of the claim's Arabic Matn is actually contained in each gold record? ----
ev_ar_set = [set(normalize_and_tokenize(e["arabic_matn"])) for e in evidence]
gold_cont = np.zeros((Q, 5))
for i, (a, g) in enumerate(zip(q_ar, gold_idx)):
    qs = set(normalize_and_tokenize(a))
    for j, d in enumerate(g):
        gold_cont[i, j] = len(qs & ev_ar_set[d]) / len(qs) if qs else 0.0
print("mean containment by gold rank (1..5):", gold_cont.mean(0).round(3))
rows = []
for th in (0.7, 0.8, 0.9, 0.95, 0.99):
    rows.append({"threshold": th, "claims with >=1 gold passing": (gold_cont >= th).any(1).mean(),
                 "top-1 gold passes": (gold_cont[:, 0] >= th).mean(), "gold IDs passing": (gold_cont >= th).mean()})
df_cont = pd.DataFrame(rows).round(3); show(df_cont)
rel_mask = gold_mask.copy()                       # gold restricted to reliable IDs
for i, g in enumerate(gold_idx):
    for j, d in enumerate(g):
        if gold_cont[i, j] < CONTAINMENT_THRESHOLD:
            rel_mask[i, d] = False
rel_rows = rel_mask.any(1)
print(f"reliable-gold subset (containment >= {CONTAINMENT_THRESHOLD}): {rel_rows.sum()} claims ({rel_rows.mean():.1%})")
AUDIT.update(mean_containment_by_gold_rank=gold_cont.mean(0).round(4).tolist(),
             reliable_gold_claims=int(rel_rows.sum()), reliable_gold_share=float(rel_rows.mean()))
df_cont.to_csv(OUT / "audit_containment.csv", index=False)

## 2. Rebuild all 11 paper systems + the 4 Bangla-only RRF systems (and save the per-claim top-10 rankings)
Rankings are stored as arrays of evidence indices. Fusion lists are the top-10 of each component, exactly as in the paper (`depth=10`); deeper fusion is studied in section 6.

In [ ]:
SYS, SYS_ORDER = {}, []
def register(name, ranks):
    SYS[name] = ranks; SYS_ORDER.append(name)

register("English BM25", fast.search([tok_exp(q) for q in q_en], k=DEEP_K))
register("Direct Bangla BM25", fast.search([tok_exp(q) for q in q_bn], k=DEEP_K))
register("Qwen 2.5-3B → English BM25", fast.search([tok_exp(q) for q in q_qwen], k=DEEP_K))
zero_bn = int((fast.scores([tok_exp(q) for q in q_bn]).max(1) == 0).sum())
print(f"Direct Bangla BM25: {zero_bn}/{Q} queries have no token in the index (ranking is tie order only)")
for n in SYS_ORDER:
    print(f"  {n:32s}", {k: round(v, 3) for k, v in metrics(SYS[n], gold_mask).items()})

In [ ]:
DENSE = {
    "bge-m3":    dict(hf="BAAI/bge-m3", pd="", pq="", name="BGE-M3"),
    "me5-large": dict(hf="intfloat/multilingual-e5-large", pd="passage: ", pq="query: ", name="mE5-Large"),
    "labse":     dict(hf="sentence-transformers/LaBSE", pd="", pq="", name="LaBSE"),
    "minilm":    dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", pd="", pq="", name="MiniLM"),
}
DOC_CHUNK, BS_DOC, BS_Q, MAXLEN_DOC, MAXLEN_Q = 2000, 16, 32, 512, 256
DE, QE = {}, {}          # document embeddings, query embeddings {"bn": ..., "en": ...}

if RUN_DENSE:
    try:
        import torch
    except ImportError:
        torch = None
    DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"
    from sentence_transformers import SentenceTransformer
    print("device:", DEVICE)
    EMB = WORK / "emb"; EMB.mkdir(exist_ok=True)
    doc_texts = [retrieval_text(e) for e in evidence]
    n_chunks = math.ceil(N_DOC / DOC_CHUNK)
    for key, cfg in DENSE.items():
        chunk_paths = [EMB / f"{key}_docs_{c}.npy" for c in range(n_chunks)]
        q_paths = {k: EMB / f"{key}_q_{k}.npy" for k in ("bn", "en")}
        if not all(p.exists() for p in chunk_paths + list(q_paths.values())):
            print("encoding with", cfg["hf"])
            kw = {"torch_dtype": torch.float16} if (torch is not None and DEVICE == "cuda") else {}
            model = SentenceTransformer(cfg["hf"], device=DEVICE, model_kwargs=kw)
            model.max_seq_length = MAXLEN_DOC
            for c, p in enumerate(chunk_paths):
                if p.exists():
                    continue
                texts = [cfg["pd"] + t for t in doc_texts[c * DOC_CHUNK:(c + 1) * DOC_CHUNK]]
                emb = model.encode(texts, normalize_embeddings=True, batch_size=BS_DOC, show_progress_bar=False)
                np.save(p, np.asarray(emb, np.float32)); print(f"  {key}: doc chunk {c + 1}/{n_chunks}")
            model.max_seq_length = MAXLEN_Q
            for kind, qs in (("bn", q_bn), ("en", q_en)):
                if not q_paths[kind].exists():
                    emb = model.encode([cfg["pq"] + q for q in qs], normalize_embeddings=True, batch_size=BS_Q, show_progress_bar=False)
                    np.save(q_paths[kind], np.asarray(emb, np.float32))
            del model; gc.collect()
            if torch is not None and DEVICE == "cuda":
                torch.cuda.empty_cache()
        DE[key] = np.vstack([np.load(p) for p in chunk_paths])
        QE[key] = {k: np.load(p) for k, p in q_paths.items()}
        assert DE[key].shape[0] == N_DOC
        print(f"{cfg['name']:10s} docs {DE[key].shape} | queries {QE[key]['bn'].shape}")

def dense_search(Qm, Dm, k=DEEP_K, chunk=256):
    out = []
    for s in range(0, len(Qm), chunk):
        S = Qm[s:s + chunk] @ Dm.T
        part = np.argpartition(-S, k - 1, axis=1)[:, :k]
        pv = np.take_along_axis(S, part, 1)
        o = np.argsort(-pv, axis=1, kind="stable")
        out.append(np.take_along_axis(part, o, 1).astype(np.int32))
    return np.vstack(out)

if RUN_DENSE:
    for key, cfg in DENSE.items():
        register(f"{cfg['name']} Direct Bangla", dense_search(QE[key]["bn"], DE[key]))

In [ ]:
def rrf_fuse_ranks(lists, depth, k_rrf=RRF_K, top=DEEP_K, weights=None):
    """RRF over per-claim ranked lists (arrays Q x D). Same tie-breaking as the paper notebook (stable sort)."""
    rows = []
    for i in range(Q):
        acc = {}
        for li, rk in enumerate(lists):
            w = 1.0 if weights is None else weights[li]
            for pos, d in enumerate(rk[i, :depth], start=1):
                acc[int(d)] = acc.get(int(d), 0.0) + w / (k_rrf + pos)
        rows.append([d for d, _ in sorted(acc.items(), key=lambda x: x[1], reverse=True)[:top]])
    W = min(len(r) for r in rows)
    return np.array([r[:W] for r in rows], np.int32)

if RUN_DENSE:
    for key, cfg in DENSE.items():
        register(f"RRF English BM25 + {cfg['name']}",
                 rrf_fuse_ranks([SYS["English BM25"], SYS[f"{cfg['name']} Direct Bangla"]], depth=TOP_K))
    # Bangla-only hybrids (S12-S15): the lexical side uses the Qwen translation of the BANGLA claim, no English claim anywhere
    for key, cfg in DENSE.items():
        register(f"RRF Qwen→BM25 + {cfg['name']}",
                 rrf_fuse_ranks([SYS["Qwen 2.5-3B → English BM25"], SYS[f"{cfg['name']} Direct Bangla"]], depth=TOP_K))
print(f"{len(SYS)} systems:", *SYS_ORDER, sep="\n  ")

# ---- save per-claim top-10 rankings (evidence IDs) + deep rankings (indices) ----
rankings_top10 = {n: {cid: [ev_ids[j] for j in SYS[n][i, :TOP_K]] for i, cid in enumerate(claim_ids)} for n in SYS_ORDER}
(OUT / "rankings_top10.json").write_text(json.dumps(rankings_top10, ensure_ascii=False), encoding="utf-8")
np.savez_compressed(OUT / "rankings_deep.npz", claim_ids=np.array(claim_ids), evidence_ids=np.array(ev_ids),
                    **{f"sys{k}": SYS[n][:, :DEEP_K] for k, n in enumerate(SYS_ORDER)})
(OUT / "system_order.json").write_text(json.dumps(SYS_ORDER, ensure_ascii=False))
print("saved rankings ->", OUT)

## 3. Reproduction check against the paper's Table II
Differences ≲ 0.005 are hardware/fp16 noise. **Direct Bangla BM25** is expected to differ (tie order, see top of notebook).

In [ ]:
paper = pd.read_csv(PAPER_TABLE).set_index("System")
rows = []
for name in paper.index:
    if name not in SYS:
        rows.append({"System": name, "status": "not computed (RUN_DENSE=False)"}); continue
    m = metrics(SYS[name], gold_mask)
    r = {"System": name}
    for c in ("R@1", "R@5", "R@10", "MRR"):
        r[f"{c} paper"], r[f"{c} here"] = paper.loc[name, c], m[c]
        r[f"d{c}"] = m[c] - paper.loc[name, c]
    r["max|d|"] = max(abs(r[f"d{c}"]) for c in ("R@1", "R@5", "R@10", "MRR"))
    r["status"] = "OK" if r["max|d|"] <= 0.005 else ("tie-order (expected)" if name == "Direct Bangla BM25" else "CHECK")
    rows.append(r)
df_repro = pd.DataFrame(rows)
show(df_repro[["System", "R@10 paper", "R@10 here", "MRR paper", "MRR here", "max|d|", "status"]].round(4))
df_repro.to_csv(OUT / "reproduction_check.csv", index=False)
for n in ("English BM25", "Qwen 2.5-3B → English BM25"):
    assert df_repro.loc[df_repro.System == n, "max|d|"].iloc[0] <= 0.002, f"{n} does not reproduce the paper"
print("BM25-based rows reproduce the paper's Table II.")

## 4. Bootstrap CIs: claim-level and Matn-clustered
The 986 claims come from fewer unique Matns (section 1d), so claim-level resampling treats related claims as independent. The clustered CI resamples whole Matn clusters. Includes every Table IV comparison **and** the RRF comparisons the paper reports without CIs.

In [ ]:
def HV(name, k, mask=None, rows=None):
    return hit_vec(SYS[name], gold_mask if mask is None else mask, k, rows)

pairs = [  # (label, baseline, proposed)
    ("Translation − Direct Bangla BM25", "Direct Bangla BM25", "Qwen 2.5-3B → English BM25"),
    ("BGE-M3 − Direct Bangla BM25", "Direct Bangla BM25", "BGE-M3 Direct Bangla"),
    ("BGE-M3 − Translation", "Qwen 2.5-3B → English BM25", "BGE-M3 Direct Bangla"),
    ("English BM25 − Translation", "Qwen 2.5-3B → English BM25", "English BM25"),
    ("LaBSE − BGE-M3", "BGE-M3 Direct Bangla", "LaBSE Direct Bangla"),
    ("mE5-Large − BGE-M3", "BGE-M3 Direct Bangla", "mE5-Large Direct Bangla"),
    ("MiniLM − BGE-M3", "BGE-M3 Direct Bangla", "MiniLM Direct Bangla"),
    ("LaBSE − Translation", "Qwen 2.5-3B → English BM25", "LaBSE Direct Bangla"),
]
for d in ("LaBSE", "BGE-M3", "mE5-Large", "MiniLM"):
    pairs += [(f"RRF[EN-BM25+{d}] − English BM25", "English BM25", f"RRF English BM25 + {d}"),
              (f"RRF[EN-BM25+{d}] − {d} alone", f"{d} Direct Bangla", f"RRF English BM25 + {d}")]
pairs += [("RRF[EN-BM25+LaBSE] − RRF[EN-BM25+BGE-M3]", "RRF English BM25 + BGE-M3", "RRF English BM25 + LaBSE")]
for d in ("LaBSE", "BGE-M3", "mE5-Large", "MiniLM"):
    pairs += [(f"RRF[Qwen→BM25+{d}] − Qwen→BM25", "Qwen 2.5-3B → English BM25", f"RRF Qwen→BM25 + {d}"),
              (f"RRF[Qwen→BM25+{d}] − {d} alone", f"{d} Direct Bangla", f"RRF Qwen→BM25 + {d}"),
              (f"RRF[Qwen→BM25+{d}] − RRF[EN-BM25+{d}] (no English query)", f"RRF English BM25 + {d}", f"RRF Qwen→BM25 + {d}")]

rows = []
for label, a, b in pairs:
    if a not in SYS or b not in SYS:
        continue
    for k in (1, 5, 10):
        va, vb = HV(a, k), HV(b, k)
        d, lo, hi, p = paired_diff_ci(va, vb)
        dc, loc, hic, pc = paired_diff_ci(va, vb, clusters=cluster_id)
        rows.append({"Comparison": label, "k": k, "diff": d, "CI_low": lo, "CI_high": hi, "p": p,
                     "clus_low": loc, "clus_high": hic, "clus_p": pc,
                     "sig(claim)": lo > 0 or hi < 0, "sig(cluster)": loc > 0 or hic < 0,
                     "CI width ratio": (hic - loc) / (hi - lo) if hi > lo else np.nan})
df_boot = pd.DataFrame(rows)
df_boot.to_csv(OUT / "bootstrap_claim_vs_cluster.csv", index=False)
show(df_boot[df_boot.k == 10].round(4))
flip = df_boot[df_boot["sig(claim)"] != df_boot["sig(cluster)"]]
print(f"\ncomparisons whose significance changes under clustering: {len(flip)}")
if len(flip):
    show(flip[["Comparison", "k", "diff", "CI_low", "CI_high", "clus_low", "clus_high"]].round(4))

## 5. Sensitivity subsets
The headline ranking is re-computed on subsets that remove or isolate the audit problems. If the ordering of systems is stable across subsets (Spearman ρ on R@10 vs. the full set), the conclusions do not hinge on the weak gold labels.

* **All**: the paper's setting.
* **Unique Matns**: first claim per source Matn (removes duplicate-source claims).
* **Reliable gold**: only gold IDs containing ≥ 90% of the claim's Arabic tokens, only claims with ≥ 1 such ID.
* **Gold has English / none**: splits claims by whether any gold record carries English text (English-query systems can only match English text).
* **Twin-aware gold**: a retrieved record with text identical to a gold record counts as a hit.

In [ ]:
first_in_cluster = np.zeros(Q, bool)
seen = set()
for i, c in enumerate(cluster_id):
    if c not in seen:
        first_in_cluster[i] = True; seen.add(c)

subsets = {
    "All": (np.ones(Q, bool), gold_mask),
    "Unique Matns": (first_in_cluster, gold_mask),
    "Reliable gold (containment)": (rel_rows, rel_mask),
    "Gold has English (any)": (cl_any, gold_mask),
    "Gold has NO English": (~cl_any, gold_mask),
    "Twin-aware gold": (np.ones(Q, bool), twin_mask),
}
rows = []
for sname, (sel, mask) in subsets.items():
    for n in SYS_ORDER:
        m = metrics(SYS[n], mask, rows=sel)
        rows.append({"Subset": sname, "n": int(sel.sum()), "System": n, **m})
df_sens = pd.DataFrame(rows)
df_sens.to_csv(OUT / "sensitivity_subsets.csv", index=False)
piv = df_sens.pivot(index="System", columns="Subset", values="R@10").loc[SYS_ORDER, list(subsets)]
print("R@10 by subset (n per subset: " + ", ".join(f"{k}={int(v[0].sum())}" for k, v in subsets.items()) + ")")
show((piv * 100).round(1).reset_index())
rho = {s: sps.spearmanr(piv["All"], piv[s])[0] for s in subsets if s != "All"}
print("\nSpearman rho of system ordering (R@10) vs. All (nan = every system scores 0 on that subset):", {k: round(float(v), 3) for k, v in rho.items()})
df_sens_mrr = df_sens.pivot(index="System", columns="Subset", values="MRR").loc[SYS_ORDER, list(subsets)]
df_sens_mrr.round(3).to_csv(OUT / "sensitivity_mrr_pivot.csv")

## 6. RRF depth ablation (+ Bangla-only fusion)
The paper fuses the **top-10** list of each component. Fusing deeper lists lets a record ranked 11–100 by both components surface. The Bangla-only rows replace the English claim with the cached Qwen translation of the Bangla claim, so no English query is needed at deployment.

In [ ]:
DEPTHS = (10, 20, 50, 100)
cfgs = []
if RUN_DENSE:
    for d in ("LaBSE", "BGE-M3", "mE5-Large", "MiniLM"):
        cfgs.append((f"EN-BM25 + {d}", ["English BM25", f"{d} Direct Bangla"], "English query"))
    for d in ("LaBSE", "BGE-M3", "mE5-Large", "MiniLM"):
        cfgs.append((f"Qwen→BM25 + {d}", ["Qwen 2.5-3B → English BM25", f"{d} Direct Bangla"], "Bangla-only"))
    cfgs.append(("LaBSE + BGE-M3 (dense only)", ["LaBSE Direct Bangla", "BGE-M3 Direct Bangla"], "control"))
rows = []
depth_runs = {}
for label, comps, kind in cfgs:
    base = None
    for depth in DEPTHS:
        r = rrf_fuse_ranks([SYS[c] for c in comps], depth=depth)
        depth_runs[(label, depth)] = r
        m = metrics(r, gold_mask)
        if depth == 10:
            base = m
        rows.append({"Fusion": label, "Query": kind, "depth": depth, **m,
                     "dR@10 vs depth10 (pp)": 100 * (m["R@10"] - base["R@10"])})
df_depth = pd.DataFrame(rows)
if len(df_depth):
    df_depth.to_csv(OUT / "rrf_depth_ablation.csv", index=False)
    show((df_depth.assign(**{c: df_depth[c] * 100 for c in ("R@1", "R@5", "R@10")}).round({"R@1": 1, "R@5": 1, "R@10": 1, "MRR": 3, "dR@10 vs depth10 (pp)": 2})))
    best_bn = df_depth[(df_depth.Query == "Bangla-only") & (df_depth.depth == 10)].sort_values("R@10", ascending=False).iloc[0]
    print(f"\nbest Bangla-only fusion at the paper's depth (10): {best_bn['Fusion']}  R@10={best_bn['R@10']:.3f}")
else:
    print("RUN_DENSE=False: skipping fusion ablation")

In [ ]:
# ---- 6b. Paper-ready Bangla-only RRF table (S12-S15), CIs are Matn-clustered bootstrap (conservative) ----
if RUN_DENSE:
    def ci10(a, b, k=10):
        return paired_diff_ci(hit_vec(SYS[a], gold_mask, k), hit_vec(SYS[b], gold_mask, k), clusters=cluster_id)
    rows = []
    for i, d in enumerate(("BGE-M3", "mE5-Large", "LaBSE", "MiniLM")):
        f, e = f"RRF Qwen→BM25 + {d}", f"RRF English BM25 + {d}"
        m = metrics(SYS[f], gold_mask)
        r = {"ID": f"S{12 + i}", "System": f"BM25 (Qwen translation of BN claim) + {d}, BN query only", **{k: m[k] for k in ("R@1", "R@5", "R@10", "MRR")}}
        for tag, base in (("vs dense alone", f"{d} Direct Bangla"), ("vs Qwen→BM25 alone", "Qwen 2.5-3B → English BM25"),
                          ("vs English-query RRF", e)):
            dm, lo, hi, _ = ci10(base, f)
            r[f"R@10 {tag} (pp)"] = 100 * dm; r[f"CI {tag}"] = f"[{100*lo:+.1f}, {100*hi:+.1f}]"
        rows.append(r)
    df_bn = pd.DataFrame(rows); df_bn.to_csv(OUT / "bangla_only_rrf_table.csv", index=False)
    show(df_bn.assign(**{c: df_bn[c] * 100 for c in ("R@1", "R@5", "R@10")}).round({"R@1": 1, "R@5": 1, "R@10": 1, "MRR": 3,
         "R@10 vs dense alone (pp)": 1, "R@10 vs Qwen→BM25 alone (pp)": 1, "R@10 vs English-query RRF (pp)": 1}))
    best = df_bn.sort_values("R@10", ascending=False).iloc[0]
    d_best = best["System"].split("+ ")[1].split(",")[0]
    best_alone = max((metrics(SYS[f"{d} Direct Bangla"], gold_mask)["R@10"], d) for d in ("BGE-M3", "mE5-Large", "LaBSE", "MiniLM"))
    print(f"\nbest Bangla-only fusion: {best['ID']} ({d_best}) R@10 = {best['R@10']:.3f}; best Bangla-only single system: {best_alone[1]} alone = {best_alone[0]:.3f}")
    print("Bangla-only fusion vs best single retriever, R@10 diff (pp) and cluster CI:",
          "%+.1f %s" % tuple([100 * ci10(f"{best_alone[1]} Direct Bangla", f"RRF Qwen→BM25 + {best_alone[1]}")[0],
                              "[%+.1f, %+.1f]" % tuple(100 * x for x in ci10(f"{best_alone[1]} Direct Bangla", f"RRF Qwen→BM25 + {best_alone[1]}")[1:3])]))
    print("Report whichever way this comes out: the claim in the paper must be the deployable (Bangla-only) number; the English-query RRF is the reference ceiling.")
else:
    print("RUN_DENSE=False: Bangla-only fusion needs the dense systems")

## 7. Failure diagnostics for the MiniLM section
The paper reports MiniLM's near-zero recall and a hub-like failure pattern but cannot say whether MiniLM simply *does not support Bangla* or suffers *hubness*. Tests:

* **Modal top-1 share** on each system's own failures, and on **shared** failures (same queries for every system, so the comparison is not confounded by different query sets).
* **Hubness**: skew of the k-occurrence distribution N₁₀ over all queries.
* **EN–BN alignment**: paired cosine vs. shuffled-pair cosine, and BN→EN translation-pair retrieval among the 986 claims. A model that cannot align Bangla with its English twin does not support Bangla; hubness is then a symptom.
* **EN-query dense retrieval**: if a model retrieves well with English queries but not Bangla, the failure is Bangla-specific, not a general corpus/hubness problem.

In [ ]:
def modal_stats(ranks, rows):
    top1 = ranks[rows, 0]
    if len(top1) == 0:
        return {"n": 0, "distinct": 0, "modal share": np.nan, "top5 share": np.nan}
    c = collections.Counter(top1.tolist()); n = len(top1)
    return {"n": n, "distinct": len(c), "modal share": c.most_common(1)[0][1] / n,
            "top5 share": sum(v for _, v in c.most_common(5)) / n}

if RUN_DENSE:
    fail = {n: ~hit_vec(SYS[n], gold_mask, 10).astype(bool) for n in SYS_ORDER}
    # (a) own failures
    rows = [{"System": n, **modal_stats(SYS[n], fail[n])} for n in SYS_ORDER]
    df_own = pd.DataFrame(rows); df_own.to_csv(OUT / "diag_modal_own_failures.csv", index=False)
    print("Modal top-1 share on each system's OWN failures"); show(df_own.round(3))
    # (b) shared failures: MiniLM fails AND the comparator fails -> same queries for both
    mini = "MiniLM Direct Bangla"
    rows = []
    for n in SYS_ORDER:
        if n == mini:
            continue
        shared = fail[mini] & fail[n]
        a, b = modal_stats(SYS[mini], shared), modal_stats(SYS[n], shared)
        rows.append({"Comparator": n, "n shared failures": a["n"], "MiniLM modal share": a["modal share"],
                     "comparator modal share": b["modal share"], "MiniLM top5": a["top5 share"], "comparator top5": b["top5 share"]})
    df_shared = pd.DataFrame(rows); df_shared.to_csv(OUT / "diag_modal_shared_failures.csv", index=False)
    print("\nModal top-1 share on SHARED failures (MiniLM fails and the comparator fails)"); show(df_shared.round(3))
    # (c) hubness: k-occurrence skew over all queries
    rows = []
    for n in SYS_ORDER:
        cnt = np.bincount(SYS[n][:, :TOP_K].ravel(), minlength=N_DOC)
        rows.append({"System": n, "skew(N10)": sps.skew(cnt), "max N10": int(cnt.max()),
                     "share of slots by 10 most-retrieved docs": np.sort(cnt)[-10:].sum() / (Q * TOP_K),
                     "docs never retrieved": float((cnt == 0).mean())})
    df_hub = pd.DataFrame(rows); df_hub.to_csv(OUT / "diag_hubness.csv", index=False)
    print("\nHubness (k-occurrence at k=10)"); show(df_hub.round(3))
else:
    print("RUN_DENSE=False: skipping dense diagnostics")

In [ ]:
if RUN_DENSE:
    rows = []
    shift = Q // 2                                   # deterministic mismatched pairing without fixed points
    for key, cfg in DENSE.items():
        E_en, E_bn = QE[key]["en"], QE[key]["bn"]
        paired = float((E_en * E_bn).sum(1).mean())
        shuffled = float((E_en * np.roll(E_bn, shift, axis=0)).sum(1).mean())
        S = E_bn @ E_en.T                            # BN -> EN translation-pair retrieval among the Q claims
        diag = np.diag(S)
        rank = 1 + (S > diag[:, None]).sum(1)
        en_ranks = dense_search(E_en, DE[key])       # English query -> evidence corpus
        m_en = metrics(en_ranks, gold_mask)
        m_bn = metrics(SYS[f"{cfg['name']} Direct Bangla"], gold_mask)
        rows.append({"Model": cfg["name"], "cos(EN,BN) paired": paired, "cos(EN,BN) shuffled": shuffled,
                     "gap": paired - shuffled, "BN->EN pair R@1": float((rank == 1).mean()), "BN->EN pair R@10": float((rank <= 10).mean()),
                     "EN-query R@10": m_en["R@10"], "BN-query R@10": m_bn["R@10"],
                     "EN-BN R@10 gap": m_en["R@10"] - m_bn["R@10"]})
    df_align = pd.DataFrame(rows); df_align.to_csv(OUT / "diag_en_bn_alignment.csv", index=False)
    show(df_align.round(3))
    print("\nHow to read this: a model with a near-zero BN->EN pair R@1 and a large EN-vs-BN query gap does not align Bangla")
    print("with English at all (Bangla unsupported); its hub-like failures are then a consequence. A model that aligns the pairs")
    print("well but still collapses onto a few records would point to hubness in the corpus instead.")

## 8. Figure data export (Fig. 4 and Fig. 5)
Exports the exact computed tables needed to plot Fig. 4 and Fig. 5 externally (no figures are generated): all systems (`S1`–`S15`, including `RRF + MiniLM` and the 4 Bangla-only RRF systems), 95% claim-level and Matn-clustered bootstrap CIs, and RRF gains over the lexical baselines.

In [ ]:
order = sorted(SYS_ORDER, key=lambda n: metrics(SYS[n], gold_mask)["R@10"])
short = lambda n: (n.replace("Direct Bangla BM25", "BN-BM25").replace(" Direct Bangla", "").replace("RRF English BM25 + ", "RRF: BM25+")
                   .replace("RRF Qwen→BM25 + ", "RRF(BN-only): Qwen+").replace("Qwen 2.5-3B → English BM25", "Qwen→BM25").replace("English BM25", "EN-BM25"))

rows = []
for idx_ord, n in enumerate(order, start=1):
    v1, v5, v10 = hit_vec(SYS[n], gold_mask, 1), hit_vec(SYS[n], gold_mask, 5), hit_vec(SYS[n], gold_mask, 10)
    _, lo, hi, _ = paired_diff_ci(np.zeros(Q), v10)
    _, clo, chi, _ = paired_diff_ci(np.zeros(Q), v10, clusters=cluster_id)
    r10 = float(v10.mean())
    rows.append({
        "order": idx_ord,
        "System": n,
        "Label": short(n),
        "n_claims": Q,
        "R@1_pct": 100.0 * float(v1.mean()),
        "R@5_pct": 100.0 * float(v5.mean()),
        "R@10_pct": 100.0 * r10,
        "R@10_CI95_low_pct": 100.0 * lo,
        "R@10_CI95_high_pct": 100.0 * hi,
        "R@10_err_low_pp": 100.0 * (r10 - lo),
        "R@10_err_high_pp": 100.0 * (hi - r10),
        "R@10_cluster_CI95_low_pct": 100.0 * clo,
        "R@10_cluster_CI95_high_pct": 100.0 * chi,
        "R@10_cluster_err_low_pp": 100.0 * (r10 - clo),
        "R@10_cluster_err_high_pp": 100.0 * (chi - r10),
    })
df_fig4 = pd.DataFrame(rows)
df_fig4.to_csv(FIG / "fig4_recall_all_systems_data.csv", index=False)
show(df_fig4[["order", "Label", "R@1_pct", "R@5_pct", "R@10_pct", "R@10_CI95_low_pct", "R@10_CI95_high_pct",
              "R@10_cluster_CI95_low_pct", "R@10_cluster_CI95_high_pct"]].round(2))
print("saved fig4 data ->", FIG / "fig4_recall_all_systems_data.csv")

In [ ]:
if RUN_DENSE:
    models = [d for d in ("LaBSE", "BGE-M3", "mE5-Large", "MiniLM")]
    en10 = 100.0 * float(hit_vec(SYS["English BM25"], gold_mask, 10).mean())
    qwen10 = 100.0 * float(hit_vec(SYS["Qwen 2.5-3B → English BM25"], gold_mask, 10).mean())
    rows = []
    for m in models:
        v_alone = hit_vec(SYS[f"{m} Direct Bangla"], gold_mask, 10)
        v_fused_en = hit_vec(SYS[f"RRF English BM25 + {m}"], gold_mask, 10)
        v_fused_bn = hit_vec(SYS[f"RRF Qwen→BM25 + {m}"], gold_mask, 10)
        d, lo, hi, _ = paired_diff_ci(hit_vec(SYS["English BM25"], gold_mask, 10), v_fused_en)
        _, clo, chi, _ = paired_diff_ci(hit_vec(SYS["English BM25"], gold_mask, 10), v_fused_en, clusters=cluster_id)
        db, lob, hib, _ = paired_diff_ci(hit_vec(SYS["Qwen 2.5-3B → English BM25"], gold_mask, 10), v_fused_bn)
        _, clob, chib, _ = paired_diff_ci(hit_vec(SYS["Qwen 2.5-3B → English BM25"], gold_mask, 10), v_fused_bn, clusters=cluster_id)
        rows.append({
            "Model": m,
            "Direct_Bangla_R@10_pct": 100.0 * float(v_alone.mean()),
            "RRF_EN_BM25_R@10_pct": 100.0 * float(v_fused_en.mean()),
            "English_BM25_baseline_R@10_pct": en10,
            "gain_vs_EN_BM25_pp": 100.0 * d,
            "gain_vs_EN_BM25_CI95_low_pp": 100.0 * lo,
            "gain_vs_EN_BM25_CI95_high_pp": 100.0 * hi,
            "gain_vs_EN_BM25_cluster_CI95_low_pp": 100.0 * clo,
            "gain_vs_EN_BM25_cluster_CI95_high_pp": 100.0 * chi,
            "RRF_BN_only_Qwen_BM25_R@10_pct": 100.0 * float(v_fused_bn.mean()),
            "Qwen_BM25_baseline_R@10_pct": qwen10,
            "gain_vs_Qwen_BM25_pp": 100.0 * db,
            "gain_vs_Qwen_BM25_CI95_low_pp": 100.0 * lob,
            "gain_vs_Qwen_BM25_CI95_high_pp": 100.0 * hib,
            "gain_vs_Qwen_BM25_cluster_CI95_low_pp": 100.0 * clob,
            "gain_vs_Qwen_BM25_cluster_CI95_high_pp": 100.0 * chib,
        })
    df_fig5 = pd.DataFrame(rows)
    df_fig5.to_csv(FIG / "fig5_rrf_gain_data.csv", index=False)
    show(df_fig5[["Model", "Direct_Bangla_R@10_pct", "RRF_EN_BM25_R@10_pct", "English_BM25_baseline_R@10_pct",
                  "gain_vs_EN_BM25_pp", "gain_vs_EN_BM25_CI95_low_pp", "gain_vs_EN_BM25_CI95_high_pp",
                  "RRF_BN_only_Qwen_BM25_R@10_pct", "gain_vs_Qwen_BM25_pp"]].round(2))
    print("saved fig5 data ->", FIG / "fig5_rrf_gain_data.csv")
else:
    print("RUN_DENSE=False: Fig. 5 data needs the dense systems")

## 9. Summary and export

In [ ]:
print("=" * 78); print("AUDIT SUMMARY"); print("=" * 78)
print(f"records: {AUDIT['n_authentic']} authentic + {AUDIT['n_fabricated']} fabricated (paper: 1000 + 1000)")
print(f"gold = BM25 top-5 of the Arabic matn: reproduced exactly for {AUDIT['gold_reproduced_exact_ordered']:.1%} of claims")
print(f"unique source matns: {AUDIT['n_unique_matns']} of {Q} claims")
print(f"claims with >=1 gold record containing >= {CONTAINMENT_THRESHOLD:.0%} of the matn: {AUDIT['reliable_gold_share']:.1%} (n={AUDIT['reliable_gold_claims']})")
print(f"English text: corpus {AUDIT['corpus_english_share']:.1%} | claims whose top-1 gold has English {AUDIT['claims_top1_gold_english']:.1%} | any gold {AUDIT['claims_any_gold_english']:.1%}")
print(f"identical-text duplicate evidence records: {AUDIT['evidence_duplicate_texts']}; claims whose gold has an out-of-gold twin: {AUDIT['claims_with_gold_twin']}")
AUDIT["spearman_vs_all"] = {k: (None if np.isnan(v) else float(v)) for k, v in rho.items()}
(OUT / "audit_summary.json").write_text(json.dumps(AUDIT, indent=2, ensure_ascii=False), encoding="utf-8")

zip_path = shutil.make_archive(str(WORK / "hmb_reanalysis_outputs"), "zip", OUT)
print("\nfiles:"); [print("  ", p.relative_to(OUT)) for p in sorted(OUT.rglob("*")) if p.is_file()]
print("\nzip:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass